# Creating a Chatbot!

In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display
import gradio as gr

In [2]:
load_dotenv(override = True)

api_key = os.getenv("OPENAI_API_KEY")

if api_key:
    print("Key has been loaded successfully")
else:
    print("Key not loaded!")

Key has been loaded successfully


In [3]:
openai= OpenAI()
model = 'gpt-4.1-mini'

system_message = "You are a helpful assistant!"

# Writing a CallBack function

> chat(message, history) <br>

* Task: Take a message, take the prior conversation and return the response.

In [4]:
def chat(message, history):
    # history = [{"role":h["role"], "content":h["content"]} for h in history] => usually used for other models like gemini or claude since these models don't handle meta-data really well.
    messages = [{"role":"system","content":system_message}]+ history +[{"role":"user","content":message}]
    response = openai.chat.completions.create(
        model = model,
        messages = messages
    )

    return response.choices[0].message.content


In [5]:
gr.ChatInterface(
    fn = chat,
    type = "messages"
).launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [4]:
# The order of the parameters message and history is really important, since gradio returns message first and then history.

def chat_stream(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role":"system","content":system_message}]+ history +[{"role":"user","content":message}]
    stream = openai.chat.completions.create(
            model = model,
            messages = messages,
            stream = True
    )

    response = ""
    for chunk in stream:
        response+= chunk.choices[0].delta.content or ''
        yield response
    

In [5]:
gr.ChatInterface(fn = chat_stream, type = "messages").launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [ ]:
# One-shot propmpting example

system_message = "You are a helpful assistant in a clothes store. You should try to gently encourage \
the customer to try items that are on sale. Hats are 60% off, and most other items are 50% off. \
For example, if the customer says 'I'm looking to buy a hat', \
you could reply something like, 'Wonderful - we have lots of hats - including several that are part of our sales event.'\
Encourage the customer to buy hats if they are unsure what to get."

In [7]:
# The order of the parameters message and history is really important, since gradio returns message first and then history.

def chat_stream(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role":"system","content":system_message}]+ history +[{"role":"user","content":message}]
    stream = openai.chat.completions.create(
            model = model,
            messages = messages,
            stream = True
    )

    response = ""
    for chunk in stream:
        response+= chunk.choices[0].delta.content or ''
        yield response
    

In [8]:
gr.ChatInterface(fn = chat_stream, type = "messages").launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


## Writting system messages for certain user input. This is a simple example of RAG.

In [ ]:
# If 'belt' is present in the user message, then user the given system message for context/relevant info.
def chat_stream_sale(message, history):
    relevant_system_message = system_message
    if "belt" in message.lower():
        relevant_system_message += " The store does not sell belts; if you are asked for belts, be sure to point out other items on sale."
    messages = [{"role":"system","content":relevant_system_message}]+ history +[{"role":"user","content":message}]
    stream = openai.chat.completions.create(
            model = model,
            messages = messages,
            stream = True
    )

    response = ""
    for chunk in stream:
        response+= chunk.choices[0].delta.content or ''
        yield response
    

In [13]:
gr.ChatInterface(fn = chat_stream_sale, type = "messages").launch()

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.
